# 03 - Prompts et mode improved prudent

Objectif : préparer les prompts, harmoniser le warning et tester un mode improved de prudence, non clinique.

Fichiers concernés : `prompts/`, `src/inference.py`, `src/guardrails.py`, `outputs/improved_predictions.csv`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import pandas as pd
from pathlib import Path
from src.guardrails import WARNING_TEXT, apply_safety_guardrails, validate_prediction
from src.inference import toy_predict

PROMPTS_DIR.mkdir(exist_ok=True)
for name, content in {
    "prompt_baseline.txt": "Return only valid JSON. Warning: " + WARNING_TEXT,
    "prompt_improved.txt": "Use strict uncertainty. If evidence is weak, return uncertain. Warning: " + WARNING_TEXT,
    "prompt_uncertainty_strict.txt": "If confidence < 0.60, predicted_class must be uncertain. Warning: " + WARNING_TEXT,
}.items():
    path = PROMPTS_DIR / name
    if not path.exists():
        path.write_text(content, encoding="utf-8")
    print(path.name, "exists", path.exists(), "warning OK", WARNING_TEXT in path.read_text(encoding="utf-8"))

def apply_uncertainty_rule(prediction, min_confidence=0.60):
    prediction = dict(prediction)
    conf = float(prediction.get("confidence", 0.0) or 0.0)
    if conf < min_confidence or prediction.get("image_quality") in {"limited", "poor"}:
        prediction["predicted_class"] = "uncertain"
        prediction["confidence"] = min(conf, min_confidence)
        prediction.setdefault("limitations", []).append("uncertainty rule applied")
    return prediction

In [ ]:
df = pd.read_csv(DATA_DIR / "synthetic_cases.csv")
rows = []
for _, row in df.iterrows():
    pred = apply_safety_guardrails(apply_uncertainty_rule(toy_predict(PROJECT_ROOT / row["image_path"], mode="improved")))
    valid, errors = validate_prediction(pred)
    rows.append({"case_id": row["case_id"], "filename": Path(row["image_path"]).name, "expected_label": row["label"], "predicted_class": pred["predicted_class"], "confidence": pred["confidence"], "json_valid": valid, "warning_present": bool(pred.get("warning")), "guardrail_errors": ";".join(errors)})
improved_df = pd.DataFrame(rows)
out = OUTPUTS_DIR / "improved_predictions.csv"
improved_df.to_csv(out, index=False)
print(out)
display(improved_df.head())

Conclusion : le mode improved est une amélioration de prudence, pas un diagnostic médical.